
# Thí nghiệm 02 — Kiến trúc các mô hình dự báo khí hậu

## 1. Tổng quan kiến trúc

### 1.1. Mục tiêu

Notebook này định nghĩa các kiến trúc được sử dụng trong những thực nghiệm dự báo chuỗi thời gian khí hậu.

Các mô hình gồm:

1. PatchTST kết hợp toán tử Koopman.
2. Autoformer kết hợp toán tử Koopman.
3. Informer kết hợp toán tử Koopman.
4. LSTM làm mô hình đối chứng.

Ba biến thể đầu sử dụng backbone để trích xuất biểu diễn tiềm ẩn, sau đó áp dụng toán tử Koopman có ràng buộc ổn định và một lớp tuyến tính để tạo dự báo.

LSTM được sử dụng làm mô hình đối chứng, không áp dụng phép biến đổi Koopman.

### 1.2. Đầu vào và đầu ra

Đầu vào của mô hình là tensor chuỗi thời gian:

$$
X\in\mathbb{R}^{B\times L\times C}
$$

Trong đó:

- $B$: số mẫu trong batch.
- $L$: độ dài chuỗi đầu vào.
- $C$: số kênh dữ liệu khí hậu.

Mô hình dự báo $H$ bước thời gian tương lai trên $C$ kênh.

Đầu ra được biểu diễn dưới dạng tensor:

$$
\hat{Y}\in\mathbb{R}^{B\times(HC)}
$$

Việc làm phẳng đầu ra giúp các mô hình sử dụng thống nhất cách tính hàm mất mát với nhãn dự báo.

### 1.3. Các thành phần chính

| Thành phần | Chức năng |
|---|---|
| Backbone | Trích xuất đặc trưng của chuỗi đầu vào |
| Koopman Operator | Biến đổi tuyến tính biểu diễn tiềm ẩn với ràng buộc ổn định |
| Linear Head | Chuyển biểu diễn tiềm ẩn thành vector dự báo |
| LSTM Baseline | Mô hình dự báo đối chứng không sử dụng Koopman |

### 1.4. Phạm vi của notebook

Notebook này chỉ định nghĩa các lớp và phương thức tính toán của mô hình.

Quá trình chuẩn bị dữ liệu, huấn luyện, tính toán kết quả và trực quan hóa được thực hiện tại các notebook riêng.

Các lớp được sử dụng chung cho nhiều nhánh thực nghiệm Climate, gồm CMIP6 Wind, Surface Pressure, ERA5 và Scenario 4 Train/Test.

**Lưu ý:** Đây là những kiến trúc dựa trên các ý tưởng của PatchTST, Autoformer và Informer, không phải bản triển khai đầy đủ các kiến trúc nguyên gốc.


In [1]:
import torch
import torch.nn as nn
import numpy as np


## 2. Toán tử Koopman có ràng buộc ổn định

### 2.1. Vai trò

Toán tử Koopman được sử dụng để mô hình hóa sự tiến hóa của biểu diễn tiềm ẩn bằng một phép biến đổi tuyến tính.

Gọi $\mathbf{z}$ là biểu diễn được backbone trích xuất, phép biến đổi Koopman được thực hiện bởi:

$$
\mathbf{z}'=K\mathbf{z}
$$

Trong đó $K$ là ma trận của toán tử Koopman.

### 2.2. Xây dựng ma trận toán tử

Lớp `StrictStableKoopmanOperator` tham số hóa ma trận dưới dạng:

$$
K=U\operatorname{diag}(s)V^\top
$$

Trong đó:

- $U$ và $V$ được xây dựng thành các ma trận trực giao bằng phép phân rã QR.
- $s$ là vector các giá trị được khống chế.
- $\operatorname{diag}(s)$ là ma trận đường chéo tạo từ vector $s$.

Các tham số có thể học được bao gồm:

- `U_raw`: ma trận tham số dùng để tạo $U$.
- `V_raw`: ma trận tham số dùng để tạo $V$.
- `S_raw`: vector tham số dùng để tạo $s$.

Các tham số này được tối ưu bằng lan truyền ngược trong quá trình huấn luyện.

### 2.3. Ràng buộc ổn định

Mỗi phần tử của $s$ được tính theo công thức:

$$
s_i=\rho_{\max}\operatorname{sigmoid}(a_i)
$$

Trong thí nghiệm:

$$
\rho_{\max}=0.99
$$

Vì:

$$
0<\operatorname{sigmoid}(a_i)<1
$$

nên:

$$
0<s_i<0.99
$$

Khi các ma trận $U$, $V$ trực giao, chuẩn phổ của toán tử được khống chế bởi:

$$
\|K\|_2=\max_i s_i<0.99
$$

Điều này bảo đảm tính co của phép biến đổi tuyến tính theo chuẩn Euclid trong điều kiện tính toán lý tưởng.

### 2.4. Đầu ra của lớp

Phương thức `forward()` trả về ba thành phần:

| Thành phần | Ý nghĩa |
|---|---|
| `z_next` | Biểu diễn tiềm ẩn sau phép biến đổi Koopman |
| `K` | Ma trận toán tử Koopman |
| `Σ` | Vector các giá trị kỳ dị được khống chế |

Ma trận và các giá trị kỳ dị có thể được sử dụng để phân tích đặc điểm ổn định của toán tử.

### 2.5. Giới hạn diễn giải

Ràng buộc chuẩn phổ của toán tử không đồng nghĩa với việc toàn bộ mô hình phi tuyến sẽ luôn dự báo chính xác.

Đây là một tính chất của lớp biến đổi Koopman, cần được phân biệt với sai số dự báo và khả năng tổng quát hóa.


In [ ]:
def _orth(w: torch.Tensor) -> torch.Tensor:
    return torch.linalg.qr(w)[0]


class StrictStableKoopmanOperator(nn.Module):
    def __init__(self, latent_dim: int, ρ_max: float = 0.99):
        super().__init__()
        self.U_raw = nn.Parameter(torch.randn(latent_dim, latent_dim))
        self.V_raw = nn.Parameter(torch.randn(latent_dim, latent_dim))
        self.S_raw = nn.Parameter(torch.randn(latent_dim))
        self.ρ_max = ρ_max

    def forward(self, z: torch.Tensor):
        U, V = _orth(self.U_raw), _orth(self.V_raw)
        Σ = torch.sigmoid(self.S_raw) * self.ρ_max
        K = U @ torch.diag(Σ) @ V.T
        return z @ K.T, K, Σ



## 3. Các thành phần hỗ trợ kiến trúc

Ba thành phần hỗ trợ được sử dụng trong những backbone gồm:

- `SinCosPosEnc`: mã hóa vị trí bằng hàm sin và cos.
- `PatchEmbed1D`: chuyển chuỗi thời gian thành các patch.
- `SeriesDecomp`: phân rã chuỗi thời gian thành thành phần xu hướng và phần dư.

### 3.1. Mã hóa vị trí Sinusoidal

`SinCosPosEnc` bổ sung thông tin về vị trí của từng phần tử trong chuỗi biểu diễn.

Với vị trí $pos$ và chiều biểu diễn $i$, mã hóa vị trí được xác định bởi:

$$
PE_{(pos,2i)}
=
\sin\left(\frac{pos}{10000^{2i/d}}\right)
$$

$$
PE_{(pos,2i+1)}
=
\cos\left(\frac{pos}{10000^{2i/d}}\right)
$$

Trong đó $d$ là kích thước biểu diễn.

Mã hóa được cộng trực tiếp vào tensor đầu vào của Transformer Encoder.

### 3.2. Patch Embedding

Lớp `PatchEmbed1D` sử dụng tích chập một chiều (`Conv1d`) để ánh xạ các đoạn liên tiếp của chuỗi thời gian sang không gian có kích thước `d_model`.

Các tham số chính:

| Tham số | Ý nghĩa |
|---|---|
| `in_ch` | Số kênh đầu vào |
| `d_model` | Kích thước embedding |
| `patch_len` | Độ dài mỗi patch |
| `stride` | Bước dịch khi tạo patch |

Trong các backbone hiện tại, `stride` được đặt bằng `patch_len`, tạo các patch không chồng lấn.

Dữ liệu được chuyển từ dạng `(B, L, C)` sang `(B, C, L)` trước khi áp dụng `Conv1d`, sau đó chuyển lại thành dạng chuỗi các token.

### 3.3. Phân rã chuỗi thời gian

Lớp `SeriesDecomp` tách chuỗi đầu vào thành:

$$
X=T+S
$$

Trong đó:

- $T$ là thành phần xu hướng (*trend*).
- $S$ là phần dư sau khi loại bỏ xu hướng, được sử dụng như thành phần mùa vụ (*seasonal*).

Xu hướng được ước lượng bằng `AvgPool1d`.

Phần dư được xác định bởi:

$$
S=X-T
$$

Trong triển khai hiện tại, kích thước kernel được truyền qua tham số `k`.

Nếu phép pooling tạo thêm một bước thời gian do padding, code cắt kết quả để khớp độ dài chuỗi gốc.

### 3.4. Vai trò trong các backbone

- PatchTST sử dụng `PatchEmbed1D` và `SinCosPosEnc`.
- Autoformer sử dụng `SeriesDecomp` và `SinCosPosEnc`.
- Informer sử dụng `PatchEmbed1D` khi bật chế độ patch, đồng thời sử dụng `SinCosPosEnc`.

Các thành phần này được dùng lại giữa các lớp nhằm tránh lặp định nghĩa và bảo đảm cách xử lý nhất quán.


In [ ]:
# 1. Mã hóa vị trí bằng hàm sin và cos
class SinCosPosEnc(nn.Module):
    def __init__(self, d_model: int, max_len: int = 10_000):
        super().__init__()
        pos = torch.arange(max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10_000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[: x.size(1)]


# 2. Chuyển chuỗi thời gian thành các patch
class PatchEmbed1D(nn.Module):
    def __init__(self, in_ch: int, d_model: int, patch_len: int, stride: int):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, d_model, kernel_size=patch_len, stride=stride)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        return x.permute(0, 2, 1)


# 3. Phân rã chuỗi thành xu hướng và phần dư
class SeriesDecomp(nn.Module):
    def __init__(self, k: int = 3):
        super().__init__()
        self.avg = nn.AvgPool1d(k, stride=1, padding=k // 2)

    def forward(self, x):
        trend = self.avg(x.transpose(1, 2)).transpose(1, 2)
        if trend.size(1) != x.size(1):
            trend = trend[:, :x.size(1)]
        return trend, x - trend



## 4. Biến thể PatchTST kết hợp Koopman

### 4.1. Mục tiêu

Biến thể này sử dụng một backbone Transformer dựa trên patch để trích xuất biểu diễn từ dữ liệu chuỗi thời gian.

Sau đó, biểu diễn được đưa qua toán tử Koopman ổn định và lớp tuyến tính để tạo dự báo.

### 4.2. PatchTST Backbone

Lớp `PatchTST_Backbone` gồm:

1. **Patch Embedding:** chia chuỗi đầu vào thành các patch và ánh xạ sang không gian `d_model`.
2. **CLS Token:** thêm một token có thể học được ở đầu chuỗi token.
3. **Positional Encoding:** bổ sung thông tin vị trí.
4. **Transformer Encoder:** mô hình hóa quan hệ giữa các token.
5. **Representation Extraction:** lấy biểu diễn của CLS token sau Encoder.

Dữ liệu được biến đổi từ:

$$
X\in\mathbb{R}^{B\times L\times C}
$$

sang một biểu diễn:

$$
z\in\mathbb{R}^{B\times d_{\text{model}}}
$$

Backbone sử dụng ba lớp Transformer Encoder theo cấu hình mặc định.

### 4.3. Kết hợp toán tử Koopman

Lớp `Koopformer_PatchTST` thực hiện:

$$
z=f_{\mathrm{PatchTST}}(X)
$$

$$
z'=Kz
$$

$$
\hat{Y}=Wz'+b
$$

Trong đó:

- $f_{\mathrm{PatchTST}}$ là backbone.
- $K$ là toán tử Koopman ổn định.
- $W$ và $b$ là tham số của lớp tuyến tính đầu ra.

Kích thước không gian tiềm ẩn của toán tử Koopman bằng `d_model`.

Đầu ra dự báo có dạng:

$$
\hat{Y}\in\mathbb{R}^{B\times(HC)}
$$

### 4.4. Giá trị trả về

Nếu `return_latents=False`, mô hình chỉ trả về dự báo.

Nếu `return_latents=True`, mô hình trả về:

`pred, z, z_next`

Ba thành phần này được sử dụng khi tính sai số dự báo và chính quy hóa ổn định trong quá trình huấn luyện.

### 4.5. Lưu ý về kiến trúc

Đây là một biến thể dựa trên PatchTST, sử dụng CLS token để tổng hợp đặc trưng.

Không mặc nhiên coi đây là bản triển khai giống hoàn toàn với PatchTST nguyên gốc, đặc biệt về cách xử lý patch và tổng hợp biểu diễn.


In [4]:

class PatchTST_Backbone(nn.Module):
    def __init__(self, input_dim: int, seq_len: int, patch_len: int, d_model: int = 48, num_layers: int = 3, num_heads: int = 4, dim_ff: int = 96):
        super().__init__()
        self.patch = PatchEmbed1D(input_dim, d_model, patch_len, patch_len)
        n_patches = int(np.ceil(seq_len / patch_len))
        self.pos = SinCosPosEnc(d_model, max_len=n_patches + 1)
        enc = nn.TransformerEncoderLayer(d_model, num_heads, dim_ff, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, x):
        x = self.patch(x)
        cls = self.cls.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.encoder(self.pos(x))
        return x[:, 0]


class Koopformer_PatchTST(nn.Module):
    def __init__(self, input_dim, seq_len, horizon, patch_len, d_model=48):
        super().__init__()
        self.backbone = PatchTST_Backbone(input_dim, seq_len, patch_len=patch_len, d_model=d_model)
        self.koop = StrictStableKoopmanOperator(d_model)
        self.fc = nn.Linear(d_model, horizon * input_dim)

    def forward(self, x, return_latents=False):
        z = self.backbone(x)
        z_next, _, _ = self.koop(z)
        pred = self.fc(z_next)
        if return_latents:
            return pred, z, z_next
        return pred



## 5. Biến thể Autoformer kết hợp Koopman

### 5.1. Mục tiêu

Biến thể này sử dụng cơ chế phân rã chuỗi thời gian để xử lý thành phần xu hướng và phần dư trước khi đưa biểu diễn qua toán tử Koopman.

Các lớp chính gồm:

- `SimpleAutoformer`.
- `Koopformer_Autoformer`.

### 5.2. Phân rã chuỗi thời gian

Đầu vào được phân rã thành:

$$
X=T+S
$$

Trong đó $T$ là xu hướng được ước lượng bằng trung bình trượt và $S$ là phần dư.

Hai thành phần được xử lý theo hai nhánh.

### 5.3. Nhánh Seasonal

Thành phần $S$ được ánh xạ sang không gian `d_model` bằng một lớp tuyến tính.

Sau đó, mô hình áp dụng:

1. Mã hóa vị trí.
2. Transformer Encoder.
3. Phép lấy trung bình theo chiều thời gian.
4. Lớp tuyến tính dự báo.

Nhánh này tạo vector có kích thước $H$.

### 5.4. Nhánh Trend

Thành phần xu hướng $T$ được làm phẳng thành vector kích thước $L\times C$.

Một lớp tuyến tính ánh xạ vector này thành đầu ra kích thước $H$.

### 5.5. Kết hợp hai nhánh

Đầu ra của `SimpleAutoformer` được xác định bằng tổng hai nhánh:

$$
z=z_{\mathrm{seasonal}}+z_{\mathrm{trend}}
$$

Vì mỗi nhánh có đầu ra kích thước $H$, biểu diễn kết hợp cũng có kích thước $H$.

### 5.6. Toán tử Koopman và dự báo

Trong `Koopformer_Autoformer`, toán tử Koopman hoạt động trên không gian tiềm ẩn có kích thước bằng `horizon`, không phải `d_model`.

Quá trình dự báo:

$$
z'=Kz
$$

$$
\hat{Y}=Wz'+b
$$

Đầu ra cuối cùng có kích thước $H\times C$ cho mỗi mẫu.

### 5.7. Giới hạn so với Autoformer gốc

Lớp `SimpleAutoformer` triển khai thành phần phân rã chuỗi và Transformer Encoder thông thường.

Nó không triển khai đầy đủ cơ chế Auto-Correlation đặc trưng của Autoformer gốc.

Vì vậy, trong báo cáo cần mô tả đây là biến thể Autoformer đơn giản hóa kết hợp Koopman, không phải một bản tái hiện đầy đủ kiến trúc Autoformer chuẩn.


In [5]:

class SimpleAutoformer(nn.Module):
    def __init__(self, input_len, horizon, input_dim, patch_len, d_model=48, num_heads=4, dim_ff=64, num_layers=3):
        super().__init__()
        self.dec = SeriesDecomp(k=patch_len)
        self.embed = nn.Linear(input_dim, d_model)
        self.pos = SinCosPosEnc(d_model, max_len=input_len)
        enc = nn.TransformerEncoderLayer(d_model, num_heads, dim_ff, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers)
        self.fc_seas = nn.Linear(d_model, horizon)
        self.fc_trend = nn.Linear(input_len * input_dim, horizon)

    def forward(self, x):
        trend, seas = self.dec(x)
        seas = self.encoder(self.pos(self.embed(seas)))
        seas_o = self.fc_seas(seas.mean(1))
        trend_o = self.fc_trend(trend.reshape(trend.size(0), -1))
        return seas_o + trend_o


class Koopformer_Autoformer(nn.Module):
    def __init__(self, input_dim, seq_len, horizon,
                 patch_len, d_model=16):
        super().__init__()
        self.backbone = SimpleAutoformer(seq_len, horizon, input_dim, patch_len=patch_len, d_model=d_model)
        self.koop = StrictStableKoopmanOperator(horizon)
        self.fc = nn.Linear(horizon, horizon * input_dim)

    def forward(self, x, return_latents=False):
        z = self.backbone(x)
        z_next, _, _ = self.koop(z)
        pred = self.fc(z_next)
        if return_latents:
            return pred, z, z_next
        return pred



## 6. Biến thể Informer kết hợp Koopman

### 6.1. Mục tiêu

Biến thể Informer sử dụng Transformer Encoder để trích xuất đặc trưng của chuỗi thời gian.

Mô hình có khả năng xử lý chuỗi đầu vào theo hai cách:

- Trực tiếp theo từng bước thời gian.
- Theo các patch được tạo bằng tích chập một chiều.

Sau quá trình trích xuất, biểu diễn được đưa qua toán tử Koopman và lớp dự báo tuyến tính.

### 6.2. Xử lý đầu vào

Trong lớp `InformerSparse`, cách xử lý phụ thuộc vào tham số `patch_len`.

Nếu:

$$
patch\_len>1
$$

mô hình sử dụng `PatchEmbed1D`.

Ngược lại, mô hình sử dụng một lớp tuyến tính để ánh xạ từng bước thời gian sang không gian `d_model`.

### 6.3. Transformer Encoder

Sau khi tạo biểu diễn đầu vào, mô hình lần lượt áp dụng:

1. Mã hóa vị trí Sinusoidal.
2. Transformer Encoder.
3. Phép trung bình theo chiều token.

Biểu diễn thu được có kích thước:

$$
z\in\mathbb{R}^{B\times d_{\text{model}}}
$$

### 6.4. Kết hợp Koopman

Lớp `Koopformer_Informer` đưa biểu diễn qua toán tử Koopman:

$$
z'=Kz
$$

Toán tử hoạt động trong không gian có kích thước `d_model`.

Lớp tuyến tính cuối cùng tạo ra:

$$
\hat{Y}\in\mathbb{R}^{B\times(HC)}
$$

Khi `return_latents=True`, mô hình trả về dự báo cùng biểu diễn trước và sau Koopman để phục vụ tính hàm mất mát trong quá trình huấn luyện.

### 6.5. Giới hạn kiến trúc

Mặc dù lớp được đặt tên `InformerSparse`, mã nguồn hiện tại sử dụng `torch.nn.TransformerEncoder`.

Kiến trúc này không triển khai cơ chế ProbSparse Attention của Informer gốc.

Vì vậy, mô hình cần được mô tả là biến thể Transformer Encoder dựa trên ý tưởng Informer, kết hợp với toán tử Koopman ổn định.


In [7]:

class InformerSparse(nn.Module):
    def __init__(self, input_dim, d_model=48, num_heads=4, dim_ff=96, num_layers=3, seq_len=120, patch_len=1):
        super().__init__()
        self.use_patch = patch_len > 1
        if self.use_patch:
            self.patch = PatchEmbed1D(input_dim, d_model, patch_len, patch_len)
            n_tokens = int(np.ceil(seq_len / patch_len))
        else:
            self.embed = nn.Linear(input_dim, d_model)
            n_tokens = seq_len
        self.pos = SinCosPosEnc(d_model, max_len=n_tokens)
        enc = nn.TransformerEncoderLayer(d_model, num_heads, dim_ff, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers)

    def forward(self, x):
        if self.use_patch:
            x = self.patch(x)
        else:
            x = self.embed(x)
        x = self.encoder(self.pos(x))
        return x.mean(dim=1)


class Koopformer_Informer(nn.Module):
    def __init__(self, input_dim, seq_len, horizon, patch_len, d_model=48):
        super().__init__()
        self.backbone = InformerSparse(input_dim, d_model=d_model, seq_len=seq_len, patch_len=patch_len)
        self.koop = StrictStableKoopmanOperator(d_model)
        self.fc = nn.Linear(d_model, horizon * input_dim)

    def forward(self, x, return_latents=False):
        z = self.backbone(x)
        z_next, _, _ = self.koop(z)
        pred = self.fc(z_next)
        if return_latents:
            return pred, z, z_next
        return pred


## 7. Mô hình đối chứng LSTM

### 7.1. Vai trò

LSTM được sử dụng làm mô hình đối chứng nhằm so sánh với ba biến thể kết hợp toán tử Koopman.

Khác với PatchTST, Autoformer và Informer, LSTM trong thí nghiệm không sử dụng toán tử Koopman.

### 7.2. Kiến trúc

Lớp `SimpleLSTMForecaster` gồm hai thành phần chính:

1. Mạng LSTM để trích xuất biểu diễn tuần tự.
2. Lớp tuyến tính để dự báo các giá trị tương lai.

Đầu vào:

$$
X\in\mathbb{R}^{B\times L\times C}
$$

Sau khi xử lý chuỗi, mô hình lấy hidden state cuối cùng của tầng LSTM trên cùng:

$$
z=h_{\mathrm{last}}
$$

Biểu diễn này được đưa qua lớp tuyến tính:

$$
\hat{Y}=Wz+b
$$

Đầu ra có kích thước:

$$
\hat{Y}\in\mathbb{R}^{B\times(HC)}
$$

### 7.3. Các tham số

| Tham số | Vai trò |
|---|---|
| `input_dim` | Số kênh dữ liệu |
| `hidden_dim` | Số chiều của hidden state |
| `num_layers` | Số tầng LSTM |
| `horizon` | Chân trời dự báo |

Các tham số cụ thể được xác định trong notebook Training của từng nhánh thực nghiệm.

### 7.4. Giao diện phục vụ huấn luyện

LSTM cũng hỗ trợ tham số `return_latents` để sử dụng chung giao diện với các biến thể khác.

Khi `return_latents=True`, mô hình trả về:

`pred, z, z`

Hai biểu diễn tiềm ẩn được trả về giống nhau, vì mô hình không có bước biến đổi Koopman.

Trong các vòng huấn luyện hiện tại, hệ số chính quy hóa ổn định được đặt bằng 0 đối với LSTM.

Vì vậy, hàm mục tiêu của LSTM chỉ sử dụng sai số dự báo.

### 7.5. Vai trò khi phân tích kết quả

Kết quả LSTM được dùng làm cơ sở đối chiếu với các biến thể có toán tử Koopman.

Tuy nhiên, sự khác biệt về sai số không thể được quy hoàn toàn cho tác động của Koopman, do các mô hình còn có những backbone và số lượng tham số khác nhau.


In [ ]:
class SimpleLSTMForecaster(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, horizon):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, horizon * input_dim)

    def forward(self, x, return_latents=False):
        _, (h_n, _) = self.lstm(x)
        z = h_n[-1]
        out = self.fc(z)
        if return_latents:
            return out, z, z
        return out